# TIP/IEF 发行人历史来源门

固定官方发行人下载各一次。只保存原字节与传输收据，不读取 SOXX 收益或构造信号。

In [1]:
import datetime,hashlib,json,pathlib,requests
root=pathlib.Path.cwd();card=root/'research/stage1_tip_ief_issuer_market_proxy_source_20261023';plan=json.loads((card/'plan.json').read_text())
raw=root/'data/snapshots/stage1_tip_ief_issuer_market_proxy_source_20261023';raw.mkdir(parents=True,exist_ok=True)
receipts={}
for symbol,url in plan['fixed_downloads'].items():
 received=datetime.datetime.now(datetime.timezone.utc).isoformat()
 try:
  response=requests.get(url,headers={'User-Agent':'SOXL-Qlib-Research source audit (read-only)'},timeout=30)
  body=response.content;(raw/(symbol+'_fund_download.response')).write_bytes(body)
  keep={k:response.headers.get(k) for k in ('Content-Type','Content-Length','Date','Last-Modified','ETag','Cache-Control') if response.headers.get(k)}
  receipts[symbol]={'url':url,'received_at_utc':received,'status_code':response.status_code,'headers':keep,'bytes':len(body),'sha256':hashlib.sha256(body).hexdigest(),'raw_path':str((raw/(symbol+'_fund_download.response')).relative_to(root)),'transport_error':None}
 except Exception as exc:
  receipts[symbol]={'url':url,'received_at_utc':received,'status_code':None,'headers':{},'bytes':0,'sha256':None,'raw_path':None,'transport_error':type(exc).__name__+': '+str(exc)}
(card/'transport_result.json').write_text(json.dumps(receipts,ensure_ascii=False,indent=2)+'\n')
print(json.dumps({k:{x:v for x,v in rec.items() if x!='url'} for k,rec in receipts.items()},ensure_ascii=False,indent=2))


{
  "TIP": {
    "received_at_utc": "2026-10-23T02:58:00.346468+00:00",
    "status_code": 200,
    "headers": {
      "Content-Type": "application/vnd.ms-excel",
      "Content-Length": "2414579",
      "Date": "Fri, 23 Oct 2026 02:58:01 GMT",
      "ETag": "W/\"11aab105\"",
      "Cache-Control": "no-cache, no-store, max-age=0, must-revalidate"
    },
    "bytes": 2414579,
    "sha256": "c40a6ac9c3fc0e2e78c691db2e02c0c56d68b4b92427febacf7c235d91c8d02c",
    "raw_path": "data/snapshots/stage1_tip_ief_issuer_market_proxy_source_20261023/TIP_fund_download.response",
    "transport_error": null
  },
  "IEF": {
    "received_at_utc": "2026-10-23T02:58:27.658320+00:00",
    "status_code": 200,
    "headers": {
      "Content-Type": "application/vnd.ms-excel",
      "Content-Length": "2486439",
      "Date": "Fri, 23 Oct 2026 02:58:29 GMT",
      "ETag": "W/\"df47dafe\"",
      "Cache-Control": "no-cache, no-store, max-age=0, must-revalidate"
    },
    "bytes": 2486439,
    "sha256": "6743

In [2]:
import hashlib, json, pathlib, re
import pandas as pd
from src.data.soxx_issuer_fund_download import read_historical
root=pathlib.Path.cwd();card=root/'research/stage1_tip_ief_issuer_market_proxy_source_20261023'
receipt=json.loads((card/'transport_result.json').read_text());frames={};audits={}
for symbol in ('TIP','IEF'):
 item=receipt[symbol];raw=root/item['raw_path']
 assert item['status_code']==200 and hashlib.sha256(raw.read_bytes()).hexdigest()==item['sha256'] and raw.stat().st_size==item['bytes']
 frame,meta=read_historical(raw);values=frame.ex_dividends.astype(str);marked=values.ne('--')
 assert values.loc[marked].map(lambda v:bool(re.fullmatch(r'[0-9]+(?:\.[0-9]+)?',v))).all()
 numeric=pd.to_numeric(values.where(marked),errors='coerce');frames[symbol]=frame
 audits[symbol]={**meta,'first_date':str(frame.index.min().date()),'last_date':str(frame.index.max().date()),'nav_positive_rows':int(frame.nav_per_share.gt(0).sum()),'ex_dividend_recorded_rows':int(marked.sum()),'ex_dividend_positive_rows':int(numeric.gt(0).sum()),'ex_dividend_zero_marker_rows':int(numeric.eq(0).sum()),'source_sha256':item['sha256'],'source_bytes':item['bytes'],'issuer_dates_2011_2022':int(frame.loc['2011-01-01':'2022-12-31'].shape[0]),'issuer_dates_2023_2026Oct22':int(frame.loc['2023-01-01':'2026-10-22'].shape[0])}
common=frames['TIP'].index.intersection(frames['IEF'].index)
result={'status':'two_official_issuer_Historical_workbooks_schema_and_date_coverage_qualified_only','plan_sha256':hashlib.sha256((card/'plan.json').read_bytes()).hexdigest(),'sources':audits,'shared_issuer_dates':len(common),'shared_issuer_dates_2011_2022':int(((common>=pd.Timestamp('2011-01-01'))&(common<=pd.Timestamp('2022-12-31'))).sum()),'shared_issuer_dates_2023_2026Oct22':int(((common>=pd.Timestamp('2023-01-01'))&(common<=pd.Timestamp('2026-10-22'))).sum()),'historical_first_public_qualified':False,'historical_revision_qualified':False,'TIP_IEF_spread_is_real_yield':False,'Qlib_calls':0,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'source_audit.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
print({k:v for k,v in result.items() if k!='sources'})


{'status': 'two_official_issuer_Historical_workbooks_schema_and_date_coverage_qualified_only', 'plan_sha256': '0e4d358329b7bff33dbad8296920998415bd979f689a003ced3db96a6fc0edcf', 'shared_issuer_dates': 5759, 'shared_issuer_dates_2011_2022': 3020, 'shared_issuer_dates_2023_2026Oct22': 957, 'historical_first_public_qualified': False, 'historical_revision_qualified': False, 'TIP_IEF_spread_is_real_yield': False, 'Qlib_calls': 0, 'SOXX_outcome_reads': 0, 'model_fits': 0, 'native_accounts': 0}


In [3]:
import datetime,hashlib,json,pathlib,requests
root=pathlib.Path.cwd();card=root/'research/stage1_tip_ief_issuer_market_proxy_source_20261023';raw=root/'data/snapshots/stage1_tip_ief_issuer_market_proxy_source_20261023'
plan=json.loads((card/'vintage_2015_pair_plan.json').read_text());receipts={}
for symbol,item in plan['samples'].items():
 received=datetime.datetime.now(datetime.timezone.utc).isoformat()
 try:
  response=requests.get(item['replay_url'],headers={'User-Agent':'SOXL-Qlib-Research vintage-source-audit (read-only)'},timeout=(5,30))
  body=response.content;path=raw/(symbol+'_2015_vintage.response');path.write_bytes(body)
  receipts[symbol]={'received_at_utc':received,'status_code':response.status_code,'content_type':response.headers.get('Content-Type'),'bytes':len(body),'sha256':hashlib.sha256(body).hexdigest(),'raw_path':str(path.relative_to(root)),'error':None}
 except Exception as exc:receipts[symbol]={'received_at_utc':received,'status_code':None,'content_type':None,'bytes':0,'sha256':None,'raw_path':None,'error':type(exc).__name__+': '+str(exc)}
(card/'vintage_2015_transport_result.json').write_text(json.dumps(receipts,ensure_ascii=False,indent=2)+'\n')
print(json.dumps(receipts,ensure_ascii=False,indent=2))


{
  "TIP": {
    "received_at_utc": "2026-10-23T03:03:05.875639+00:00",
    "status_code": 200,
    "content_type": "application/vnd.ms-excel;charset=UTF-8",
    "bytes": 1176004,
    "sha256": "edd0253d521dc703164a4f9626209b9ab7e3e111ebbf5460f1092dbc00b7acd0",
    "raw_path": "data/snapshots/stage1_tip_ief_issuer_market_proxy_source_20261023/TIP_2015_vintage.response",
    "error": null
  },
  "IEF": {
    "received_at_utc": "2026-10-23T03:03:10.443267+00:00",
    "status_code": 200,
    "content_type": "application/vnd.ms-excel;charset=UTF-8",
    "bytes": 1274009,
    "sha256": "3697687285ef623b240e683756794609912688628d988a0051b378926a1ed861",
    "raw_path": "data/snapshots/stage1_tip_ief_issuer_market_proxy_source_20261023/IEF_2015_vintage.response",
    "error": null
  }
}


In [4]:
import collections, datetime, hashlib, json, pathlib, re
from decimal import Decimal
from xml.etree import ElementTree as ET
from src.data.soxx_issuer_fund_download import read_historical
root=pathlib.Path.cwd();card=root/'research/stage1_tip_ief_issuer_market_proxy_source_20261023';raw=root/'data/snapshots/stage1_tip_ief_issuer_market_proxy_source_20261023'
transport=json.loads((card/'vintage_2015_transport_result.json').read_text());saved=json.loads((card/'vintage_2015_result.json').read_text());precision=json.loads((card/'vintage_2015_dividend_precision_result.json').read_text());ns='{urn:schemas-microsoft-com:office:spreadsheet}'
observed={}
for symbol in ('TIP','IEF'):
 item=transport[symbol];path=root/item['raw_path'];assert hashlib.sha256(path.read_bytes()).hexdigest()==item['sha256']
 text=path.read_text('utf-8-sig');assert text.count('</Style>')==1;text=text.replace('</Style>','</ss:Style>')
 text=re.sub(r'&(?!#(?:\d+|x[0-9A-Fa-f]+);|amp;|lt;|gt;|quot;|apos;)','&amp;',text)
 tree=ET.fromstring(text);sheet=next(x for x in tree.findall(ns+'Worksheet') if x.get(ns+'Name')=='Historical')
 rows=[[e.text for e in row.findall('.//'+ns+'Data')] for row in sheet.findall('.//'+ns+'Row')];header=rows[0]
 assert header==['As Of','Index Level','NAV per Share','Ex-Dividends']
 old={datetime.datetime.strptime(row[0],'%b %d, %Y').date():(row[2],row[3]) for row in rows[1:]}
 current,_=read_historical(raw/(symbol+'_fund_download.response'));nav_bad=[];div_bad=[]
 for day,(nav,div) in old.items():
  row=current.loc[str(day)];current_div=str(row.ex_dividends)
  if Decimal(nav)!=Decimal(str(row.nav_per_share)):nav_bad.append(day)
  if div!=current_div and div not in ('--','') and current_div not in ('--','') and Decimal(div)!=Decimal(current_div):div_bad.append((day,Decimal(current_div)-Decimal(div)))
 assert len(old)==saved['sources'][symbol]['overlap_rows'] and len(nav_bad)==saved['sources'][symbol]['nav_mismatch_count']==0
 assert len(div_bad)==saved['sources'][symbol]['ex_dividend_mismatch_count']
 observed[symbol]={'overlap':len(old),'NAV_mismatches':len(nav_bad),'ex_dividend_numeric_mismatches':len(div_bad)}
 if symbol=='IEF':
  assert dict(sorted(collections.Counter(day.year for day,_ in div_bad).items()))=={int(k):v for k,v in precision['by_year'].items()}
  assert max(abs(value) for _,value in div_bad)==Decimal(precision['maximum_absolute_decimal_difference'])
print(observed)


{'TIP': {'overlap': 2832, 'NAV_mismatches': 0, 'ex_dividend_numeric_mismatches': 0}, 'IEF': {'overlap': 3179, 'NAV_mismatches': 0, 'ex_dividend_numeric_mismatches': 37}}


## 中文结论

两份官方原件各一次 GET，TIP/IEF 共用 5759 日；2011–2022 共用3020日，2023–2026-10-22 共用957日。NAV 无缺值，TIP 除息列73个明确零标记。原字节已 NAS 归档并定向恢复。当前回溯工作簿不能证明逐日首次公开或修订历史，TIP/IEF 差不是官方10年实际收益率；0 Qlib、0标签、0拟合、0账户。
2015 年原件对照：TIP 2832、IEF 3179 个重叠日 NAV 完全一致，TIP 除息完全一致；IEF 37个小额差异仅2002–2005，2011–2015 无差异。旧原字节另在 NAS 冷恢复通过；更晚训练年与逐日首发仍未知。